# 球座標のホッジラプラシアン

球座標 $(r,\theta,\phi)$ におけるユークリッド計量は

$$ds^2=dr^2+r^2d\theta^2+r^2\sin^2\theta\,d\phi^2.$$

です。この計量から $d$、$\star$、余微分を構成します。以下で採用する
余微分の規約では、スカラーに対する結果は通常の座標表示による
正符号のラプラス作用素に負号を付けたものになります。


## 球座標の計量

計量の行列式は $r^4\sin^2\theta$ です。一方、逆計量には角度方向の
尺度因子 $r^{-2}$ と $(r^2\sin^2\theta)^{-1}$ が含まれます。


In [1]:
declare symbol r, θ, φ : MathValue

def N : Integer := 3
def x : Vector MathValue := [| r, θ, φ |]

def g_i_j : Matrix MathValue :=
  [| [| 1, 0, 0 |]
   , [| 0, r ^ 2, 0 |]
   , [| 0, 0, r ^ 2 * (sin θ) ^ 2 |] |]_i_j
def g~i~j : Matrix MathValue :=
  [| [| 1, 0, 0 |]
   , [| 0, 1 / r ^ 2, 0 |]
   , [| 0, 0, 1 / (r ^ 2 * (sin θ) ^ 2) |] |]~i~j


In [2]:
g_#_#


$\begin{pmatrix} 1 & 0 & 0 \\ 0 & r^{2} & 0 \\ 0 & 0 & \sin(θ)^{2} r^{2} \\ \end{pmatrix}_{\#\#}^{\;\;}$

## 微分形式の演算子

ホッジスターは $g^{ij}$ で形式の添字を上げ、3次元の
レヴィ・チヴィタテンソルと縮約します。


In [3]:
def d (A : Tensor MathValue) : Tensor MathValue :=
  !(flip ∂/∂) x A

def hodge (A : DiffForm MathValue) : DiffForm MathValue :=
  let k := dfOrder A
   in withSymbols [i, j]
        sqrt (abs (M.det g_#_#)) *
        foldl
          (.)
          ((ε' N k)_(i_1)..._(i_N) . A..._(j_1)..._(j_k))
          (map (\n -> g~(i_n)~(j_n)) [1..k])

def δ (A : DiffForm MathValue) : DiffForm MathValue :=
  let k := dfOrder A
   in ((-1) ^ (N * (k + 1) + 1)) * hodge (d (hodge A))


ホッジラプラシアンは $d\delta+\delta d$ です。スカラーと体積形式には、
より短い端点の公式を用います。


In [4]:
def Δ (A : DiffForm MathValue) : DiffForm MathValue :=
  match dfOrder A as integer with
  | #0 -> δ (d A)
  | #N -> d (δ A)
  | _  -> d (δ A) + δ (d A)

def f : MathValue := function (r, θ, φ)


In [5]:
Δ f


$-\frac{\partial^2 f}{\partial 1^2} - 2 \frac{\partial f}{\partial 1} r^{-1} - \frac{\partial^2 f}{\partial 2^2} r^{-2} - \cos(θ) \sin(θ)^{-1} \frac{\partial f}{\partial 2} r^{-2} - \sin(θ)^{-2} \frac{\partial^2 f}{\partial 3^2} r^{-2}$

式を簡約すると、出力は

$$
\Delta f=-\left(
f_{rr}+\frac{2}{r}f_r
+\frac1{r^2}f_{\theta\theta}
+\frac{\cos\theta}{r^2\sin\theta}f_\theta
+\frac1{r^2\sin^2\theta}f_{\phi\phi}
\right).
$$

となります。座標に依存する各係数はすべて計量から導かれており、
球座標のラプラシアンの公式そのものは計算への入力として使っていません。
